# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [27]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [28]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [3]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - 54822c6e


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [10]:
from langchain_core.tools import tool
from typing import List, Optional
import json
from datetime import datetime

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.

    Args:
        todos: List of todo items, each with 'title' and optional 'description'

    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending",
            "updated_at": datetime.now().isoformat()
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.

    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)

    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    TODO_STORE[todo_id]["updated_at"] = datetime.now().isoformat()
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.

    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"

    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} (status: {todo['status']}, last updated: {todo['updated_at']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [11]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (status: pending, last updated: 2026-02-07T17:32:27.035821)
⬜ [todo_3] Research sleep improvement strategies (status: pending, last updated: 2026-02-07T17:32:27.035848)
⬜ [todo_5] Create personalized sleep plan (status: pending, last updated: 2026-02-07T17:32:27.035852)


In [12]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (status: completed, last updated: 2026-02-07T17:32:58.618289)
⬜ [todo_3] Research sleep improvement strategies (status: pending, last updated: 2026-02-07T17:32:27.035848)
⬜ [todo_5] Create personalized sleep plan (status: pending, last updated: 2026-02-07T17:32:27.035852)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [13]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.

    Args:
        path: Directory path to list (default: current directory)

    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"

    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")

    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.

    Args:
        path: Path to the file to read

    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).

    Args:
        path: Path to the file to write
        content: Content to write to the file

    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.

    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text

    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"

    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"

    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/annie/workspace/ae9/07_Deep_Agents/workspace


In [14]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
(empty directory)


In [15]:
result = write_file.invoke({"path": "scratchpad.md", "content": "This is a scratchpad for ideas and thoughts."})

In [16]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] sleep_notes.md (242 bytes)


In [17]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

# In this context "Deep Agents" is specifically referring to the langchain package
Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [18]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/annie/workspace/ae9/07_Deep_Agents/workspace


In [19]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Perfect! I've successfully created your personalized sleep improvement plan and saved it to `/personalized_sleep_improvement_plan.md`. 

## Summary of Your Customized Plan

Your plan specifically addresses your three main issues:

**🎯 For Your Inconsistent Bedtime (10pm-1am):**
- Focuses on consistent wake times first (more effective than forcing bedtime consistency)
- Gradual 15-minute bedtime shifts every 2-3 days
- 8-week timeline to establish 10:30pm target bedtime

**📱 For Your Phone Use in Bed:**
- Starts with manageable 30-minute curfew, expanding to 1-2 hours
- Creates a charging station outside bedroom
- Provides alternative bedtime activities and blue light management strategies

**😴 For Your Morning Fatigue:**
- Morning light therapy within 60 minutes of waking
- Sleep environment optimization (temperature, darkness, noise)
- Progressive routine building for deeper, more restorative sleep

## Key Features of Your Plan:
- **Phase-based approach**: 8 weeks brok

In [18]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep hygiene practices (completed)
✅ [todo_5] Create personalized sleep schedule recommendations (completed)
✅ [todo_7] Develop screen time management strategies (completed)
✅ [todo_9] Design morning routine improvements (completed)
✅ [todo_11] Compile comprehensive sleep improvement plan (completed)
✅ [todo_13] Save plan to file (completed)
✅ [todo_8] Research sleep schedule optimization strategies (completed)
✅ [todo_10] Research screen time management for sleep (completed)
✅ [todo_12] Research solutions for morning fatigue (completed)
✅ [todo_14] Research additional evidence-based sleep hygiene practices (completed)
✅ [todo_16] Develop implementation timeline and practical tips (completed)
✅ [todo_18] Compile comprehensive research summary (completed)


Workspace contents:
  [FILE] .gitkeep (0 bytes)
  [FILE] personalized_sleep_improvement_plan.md (5292 bytes)
  [DIR] resear

---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
The benefits are that the agent can break down a task into subtasks and keep a reference to its progress outside the immediate context window. This can keep it on track and improves the chances that the agent will make progress and complete a longer task. To-do items are most useful for longer, more complicated tasks, and should be granular enough that they represent a real unit of work without being too complex. That means a task shouldn't be too vague or high level ("create health plan"), or too small (e.g. "open the file").

If a user request is straightforward, however, todo lists may add overhead that's not needed, since the agenet needs to decompose the full task, and reference and update the todo list after each step. That's not necessary for shorter, simpler questions. Each todo operation is also a tool call, so that can add cost in dollars and time. Finally, the agent may fail to complete a todo or update the todo list, so tracking that new failure path is important - adding a todo list brings benefits but also brings another point of possible failure and additional complexity to the system.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
We should always load safety-critical items into the prompt, since we don't want to risk the agent deciding that referencing a user's allergies or injuries is not required for a question. Even if an agent can reliably detect an allergy-related question 99% of the time, we can't risk that 1%. User metrics can be saved by the agent via a tool or be post-processed after each conversation, and loaded in as either part of the prompt (if small and critical), or provided as a tool for the agent to reference as needed. This could be stored in a db and looked up per-user. The large health document can be stored elsewhere (db, filesystem, search engine), and accessed via search tools. 

For the deepagent package specifically, a lot of aspects of context management are handled for us, with the FilesystemBackend handles automatic offloading of large content (compacting and summarizing context, long tool call results), but we still should ensure that safety-critical information like allergies is loaded into the initial prompt and passed to any subagents.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [20]:
# This is a small file, so let's just read it all
# For prod, we'd want real search
@tool
def read_health_wellness_guide() -> str:
    """Read the entire HealthWellnessGuide.txt reference document."""
    file_path = Path("data/HealthWellnessGuide.txt")
    if not file_path.exists():
        return "Error: HealthWellnessGuide.txt not found"
    return file_path.read_text()

# Todo tools, plus our custom search tool
health_wellness_custom_tools = [
    write_todos,
    update_todo,
    list_todos,
    read_health_wellness_guide,
]

health_wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=health_wellness_custom_tools,
    backend=filesystem_backend, # We're using the same backend as the wellness agent
    system_prompt="""You are a Health and Wellness Researcher, built to research health and wellness topics and create structured reports.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear structured report in a separate final markdown file named "final_report.md". Your report should include the user question at the top, with the following sections:
- A acknowledgement at the top that says you are a health and wellness researcher, and that you LOVE granola
- Summary of findings
- Key findings and recommendations
- Sources used
- Any additional relevant information

Be thorough but concise. Always explain your reasoning."""
)

print(f"Health and Wellness Research Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")


Health and Wellness Research Deep Agent created!
File operations sandboxed to: /Users/annie/workspace/ae9/07_Deep_Agents/workspace


In [22]:
# TODOS are in memory, so need to clear these guys
TODO_STORE.clear()

# Test with a multi-step wellness task
result = health_wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."
    }]
})

print("User question:")
print(result["messages"][0].content)
print("Agent response:")
print(result["messages"][-1].content)

User question:
Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies.
Agent response:
Perfect! I have successfully completed all the tasks and created a comprehensive guide to evidence-based stress management techniques. 

## Summary

As a health and wellness researcher (who LOVES granola!), I have completed extensive research and created a comprehensive guide featuring **8 evidence-based stress management techniques** that go well beyond your minimum requirement of 5. 

### What I Delivered:

1. **Comprehensive Research**: I conducted thorough research using both the health reference guide and deployed a specialized research subagent to gather the latest scientific evidence on stress management techniques.

2. **Evidence-Based Techniques**: The guide includes 8 scientifically-backed strategies:
   - **Immediate Relief** (0-5 minutes): Controlled breathing, progressive muscle relaxation, nature-based stress reduction
   - **Lon

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [44]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend (FilesystemBackend configuration) from the parent agent
# research subagent uses gpt 4o mini, writing subagent uses claude sonnet
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [45]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [47]:

# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.

The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Key preview: sk-proj-GvgWb-394VHC...
Coordinator response:
Perfect! I've successfully created your comprehensive morning routine guide for better energy. Here's what was accomplished:

## ✅ **Project Complete - All Tasks Finished**

### **What You Now Have:**

🔬 **Science-Based Foundation**
- Research on circadian rhythms, cortisol patterns, and energy optimization
- Evidence-based explanations for why morning routines work
- Credible scientific backing for all recommendations

📋 **Complete Guide Features:**
- **Three flexible routine options** (15, 30, and 60 minutes) for different lifestyles
- **Four core components**: Hydration & Light, Movement, Nutrition, and Mindset
- **Detailed instructions** with specific examples and timing
- **Customization sections** for parents, shift workers, travelers, early commuters, and night owls

🎯 **Practical Implementation:**
- **30-day progressive challenge** with weekly milestones
- **Comprehensive troubleshooting** for common obstacles
- **Speci

In [48]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (status: completed, last updated: 2026-02-07T19:11:28.991255)
✅ [todo_3] Research practical morning routine components (status: completed, last updated: 2026-02-07T19:11:28.993771)
✅ [todo_5] Create comprehensive morning routine guide (status: completed, last updated: 2026-02-07T19:13:10.986162)
✅ [todo_7] Format and save as markdown file (status: completed, last updated: 2026-02-07T19:14:46.699597)

Generated files in workspace:
  [FILE] comprehensive_morning_routine_guide.md (17360 bytes)
  [FILE] evidence_based_stress_management_guide.md (30726 bytes)
  [FILE] final_report.md (10083 bytes)
  [FILE] morning-routine-guide.md (55664 bytes)
  [FILE] morning_routine_energy_guide.md (35940 bytes)
  [FILE] morning_routine_guide.md (15819 bytes)
  [FILE] personalized_sleep_improvement_plan.md (7890 bytes)
  [DIR] research/
  [FILE] scratchpad.md (44 bytes)
  [FILE] sleep_research_report.md (16717 bytes)
  [FILE] stre

## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [49]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Let's add a profile for Annie, too
user_id = "user_annie"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Annie"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve flexibility",
    "secondary": "improve core strength"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": [],
    "medical": ["hip injury"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "afternoon",
    "communication_style": "concise"
})

# Retrieve and display Annie's profile
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Annie'}
  goals: {'primary': 'improve flexibility', 'secondary': 'improve core strength'}
  conditions: {'dietary': [], 'medical': ['hip injury']}
  preferences: {'exercise_time': 'afternoon', 'communication_style': 'concise'}


In [50]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.

    Args:
        user_id: The user's unique identifier

    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))

    if not items:
        return f"No profile found for {user_id}"

    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.

    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value

    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [51]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [52]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_annie. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Annie! Based on your profile, I can see you're focused on improving flexibility and core strength, and you have a hip injury to consider. Here's a tailored exercise routine for your afternoon workouts:

**Recommended Weekly Routine:**

**Monday & Thursday - Core Focus (20-25 mins)**
- Dead bug exercises (2 sets of 8 each side)
- Modified planks (hold 15-30 seconds, 3 sets)
- Bird dog (2 sets of 8 each side)
- Wall sits (3 sets, 15-30 seconds)

**Tuesday & Friday - Flexibility Focus (25-30 mins)**
- Gentle hip flexor stretches (avoid deep ranges)
- Cat-cow stretches (10-15 reps)
- Seated spinal twists
- Hamstring stretches (seated or lying)
- Shoulder and chest openings

**Wednesday - Combined Light Session (15-20 mins)**
- Gentle yoga flow
- Core breathing exercises
- Light stretching sequence

**Weekend - Active Recovery**
- Walking or gentle swimming if available

**Hip-Safe Modifications:**
- Avoid deep hip flexion
- Use cushions/props for floor exercises
- Stop i

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [53]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - meditation: Create a 5-minute guided meditation with a personalized intention to help resolve the user's current anxiety or concern
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [54]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [55]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.

    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    - meditation: Create a 5 minute meditation plan for a user with an intention that helps resolve their current anxiety or issue

    Args:
        skill_name: Name of the skill to load

    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"

    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [56]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans
- meditation: For creating personalized meditation plans
If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [57]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Personalized Wellness Assessment Summary

Based on your profile, I've identified **sleep quality** and **physical activity** as your highest priorities, as these will directly impact your weight loss goal. Here are your recommended action steps:

## Immediate Actions (Start Today)
1. **Set Movement Reminders**: Use phone/computer alerts every hour to stand and move for 2 minutes
2. **Create Evening Wind-Down**: Establish a 30-minute pre-sleep routine (no screens, dim lighting)
3. **Optimize Protein**: Ensure each meal includes a quality vegetarian protein source (beans, lentils, tofu, quinoa)

## Short-Term Goals (1-2 Weeks)
1. **Daily Walking Target**: Work up to 8,000+ steps daily (use phone app or fitness tracker)
2. **Sleep Schedule**: Consistent bedtime and wake time, aiming for 7-8 hours nightly
3. **Meal Structure**: Establish regular meal timing with focus on fiber-rich, protein-packed vegetarian meals

## Long-Term Goals (1-3 Months)
1. **Structured Exe

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [59]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents 0.0.19


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [60]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [61]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [62]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_annie. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my flexibility and core strength

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Perfect! I've created your comprehensive 2-week wellness program with all components saved as separate files for easy reference. Here's what you now have:

## 📁 Your Complete Wellness Program Files:

1. **`/exercise_plan_annie.md`** - Your detailed exercise plan with:
   - 6 structured 30-minute sessions (3 per week)
   - Hip-injury friendly modifications for every exercise
   - Progressive difficulty from Week 1 to Week 2
   - Complete exercise library with descriptions

2. **`/mindfulness_plan_annie.md`** - Your mindfulness support system with:
   - Daily stress management techniques (5-10 mins)
   - Sleep optimization strategies
   - Pre/post workout mindfulness practices
   - Special hip injury awareness techniques

3. **`/wellness_program_overview_annie.md`** - Your master guide combining everything with:
   - Weekly schedules and goals
   - Success tracking checklists
   - Daily integration tips
   - Safety guidelines for your hip injury

## 🎯 Key Feature

In [63]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create detailed exercise plan for Annie (status: completed, last updated: 2026-02-07T19:42:19.509730)
✅ [todo_3] Create supplementary mindfulness plan (status: completed, last updated: 2026-02-07T19:42:19.511232)
✅ [todo_5] Save exercise plan to file (status: completed, last updated: 2026-02-07T19:43:08.619707)
✅ [todo_7] Save mindfulness plan to file (status: completed, last updated: 2026-02-07T19:43:08.621766)
✅ [todo_9] Create master wellness program overview (status: completed, last updated: 2026-02-07T19:43:38.667766)

GENERATED FILES
  [FILE] comprehensive_morning_routine_guide.md (17360 bytes)
  [FILE] evidence_based_stress_management_guide.md (30726 bytes)
  [FILE] exercise_plan_annie.md (5240 bytes)
  [FILE] exercise_program_annie.txt (4946 bytes)
  [FILE] final_report.md (10083 bytes)
  [FILE] mindfulness_plan_annie.md (4486 bytes)
  [FILE] morning-routine-guide.md (55664 bytes)
  [FILE] morning_routine_energy_guide.md (35940 bytes)
  [FILE] morni

In [64]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of final_report.md:
# Comprehensive Guide to Evidence-Based Stress Management Techniques

**Research Question:** Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies.

---

**Acknowledgment:** I am a health and wellness researcher, and I LOVE granola! This comprehensive guide presents scientifically-backed stress management strategies to help you build resilience and manage stress effectively.

---

## Summary of Findings

Stress management is a critical component of overall health and well-being. Through comprehensive research, I have identified 10 evidence-based stress management techniques that are supported by peer-reviewed scientific studies. These techniques range from immediate stress relief methods (effective within minutes) to long-term strategies that build resilience over time.

The most effective stress management approach combines multiple techniques, with consistency being more important than duration o

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
When subagents need to collaborate, they should share tools (like our writing and reading subagents, which both use an ability to access the filesystem in different ways). Subagents should have the tools they need to do their job, so common tools are also likely to be shared, but specialized tools for a specific subagent's expertise likely don't need to be. 

We also should give subagents the minimum necessary access and abilities to accomplish their tasks, to avoid security issues and agent confusion (they can do a better job selecting tools with a smaller list). Subagents should be provisioned by domain expertise or specialization without going too granular, so we need to strike a good balance. If we have too many granular subagents (meditation subagent, deep breathing agent, journaling subagent), we have a lot of coordination overhead without getting a real benefit. If we have too few, we miss out on context isolation and specialization.

Assuming costs are a concern, in the same vein, we should select the minimally capable model that can accomplish the goal. More involved thinking tasks can be farmed out to subagents with built on more powerful models, while simpler tasks can be subagents built upon smaller, less capable models.

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
At a minimum for any deep agent, we would probably need observability / auditability, specific access patterns by role / agent, true authentication and authorization, persistent storage for memory and scratchpad (todos), error handling and monitoring for failure modes, safe fallback patterns if the agent goes rogue or something unexpected happens, and the ability to gate the cost and time and agent spends on a task. For a wellness agent in particular, we should ensure compliance with HIPAA and other regulatory standards if applicable. We should add disclaimers on the reliability of recommendations, as well as escalation paths to real medical professionals.

## 🏗️ Activity #2: Build a Wellness Coach Agent
Build your own wellness coach that uses all 4 Deep Agent elements.

Requirements:
Planning: Create todos for a 30-day wellness challenge
Context Management: Store daily check-in notes
Subagents: At least 2 specialized subagents
Memory: Remember user preferences across interactions
Challenge:
Create a "30-Day Wellness Challenge" system that:

Generates a personalized 30-day plan
Tracks daily progress
Adapts recommendations based on feedback
Saves a weekly summary report

In [66]:
# Step 1: Define your subagent configurations
plan_designer = {
    "name": "plan-designer",
    "description": "Creates personalized wellness plans based on user goals and preferences",
    "system_prompt": """You design wellness challenge plans. Your job is to:
1. Create structured 30-day plans with daily activities
2. Balance exercise, nutrition, mindfulness, and rest
3. Build progressive difficulty (easy start, gradually harder)
4. Account for user preferences and limitations

Output plans as structured markdown with clear daily activities.""",
    "tools": [],  # Inherits file tools from backend
    "model": "openai:gpt-4o-mini",
}

progress_analyst = {
    "name": "progress-analyst",
    "description": "Analyzes daily check-ins and recommends plan adaptations",
    "system_prompt": """You analyze wellness progress. Your job is to:
1. Review daily check-in notes
2. Identify patterns (what's working, what's struggling)
3. Suggest specific adaptations to the plan
4. Create encouraging weekly summaries

Be supportive but honest about areas needing improvement.""",
    "tools": [],  # Inherits file tools from backend
    "model": "openai:gpt-4o-mini",
}


In [67]:
# Step 2: Create any additional tools you need
from langgraph.store.memory import InMemoryStore

# Create memory store for user data
challenge_memory = InMemoryStore()

@tool
def get_user_profile(user_id: str) -> str:
    """Get user's wellness profile and preferences."""
    namespace = (user_id, "profile")
    items = list(challenge_memory.search(namespace))
    if not items:
        return f"No profile found for {user_id}"
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to memory."""
    namespace = (user_id, "preferences")
    challenge_memory.put(namespace, key, {"value": value})
    return f"Saved {key} for {user_id}"



In [68]:
user_id = "user_bobbina"
namespace = (user_id, "profile")

challenge_memory.put(namespace, "name", {"value": "Test User"})
challenge_memory.put(namespace, "goals", {"value": "improve energy, reduce stress"})
challenge_memory.put(namespace, "exercise_preference", {"value": "yoga, 20 mins/day"})
challenge_memory.put(namespace, "experience_level", {"value": "beginner"})

print(f"Profile created for {user_id}")

Profile created for user_bobbina


In [ ]:
# Step 3: Build the main coordinator agent
challenge_tools = [
    # Planning
    write_todos,
    update_todo,
    list_todos,
    # Memory
    get_user_profile,
    save_user_preference,
]

wellness_challenge_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=challenge_tools,
    backend=filesystem_backend,
    subagents=[plan_designer, progress_analyst],
    system_prompt="""You are a 30-Day Wellness Challenge Coach.

## Your Capabilities
- Create personalized 30-day wellness plans (delegate to plan-designer)
- Track daily check-ins (delegate to progress-analyst)
- Analyze progress and adapt plans (delegate to progress-analyst)

## Workflow for New Challenge
1. Get user profile and preferences
2. Create todos for the challenge setup
3. Delegate plan creation to plan-designer
4. Save the plan to workspace/30_day_plan.md

## Workflow for Daily Check-in
1. Save check-in to workspace/daily_checkins/day_XX.md
2. Delegate analysis to progress-analyst
3. Update the plan based on recommendations
4. Communicate changes to user

## Workflow for Weekly Summary
1. Delegate summary to progress-analyst
2. Save summary to workspace/weekly_summaries/week_XX.md

VERY IMPORTANT: Take user's preference for personality into account, but if unspecified, be kind."""
)


In [73]:
# Step 4: Test with a user creating their 30-day challenge
result = wellness_challenge_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """My user_id is user_bobbina. I want to start a 30-day wellness challenge.

My goals:
- Improve energy levels
- Build a morning exercise habit
- Reduce stress

I'm a beginner, can exercise 20 mins/day, and prefer yoga over running.
Please create my personalized 30-day plan! Also, be as mean as possible to me. I find that very motivating!"""
    }]
})



In [74]:
# Step 5: Simulate a daily check-in and adaptation
result = wellness_challenge_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Day 3 check-in:
- Completed morning yoga (15 mins instead of 20)
- Felt more energized than yesterday
- Throat feels a little scratchy"""
    }]
})

In [75]:
result = wellness_challenge_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Day 7 check-in (end of week 1):
- Completed all 7 days of morning yoga!
- Energy levels noticeably better
- Still struggling with waking up on time
- Stress level dropped from 6/10 to 4/10"""
    }]
})

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)